In [ ]:
from tld import get_fld
from urllib.parse import urlparse
import json
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import random

Get all the entries that were not blocked

In [ ]:
def get_valid_entries(har):
    valid = []
    for entry in har['log']['entries']:
        if entry['response']['status'] <= 0:
            continue
        if entry['response'].get('_failureText'):
            continue
        valid.append(entry)
    return valid

Get the number of valid entries

In [ ]:
def get_number_of_valid_entries(valid_entries):
    return len(valid_entries)

Get the number of third party requests per website

In [ ]:
def is_third_party(page_url, request_url):
    request_domain = get_fld(request_url, fail_silently=True)
    if request_domain is None:
        return False
    return get_fld(page_url) != request_domain

def getNumThirdPartyRequests(entries, page_url):
    count = 0
    for entry in entries:
        req_url = entry['request']['url']
        if is_third_party(page_url, req_url):
            count += 1
    return count

Get the number of third-party domains per website

In [ ]:
def get_third_party_domains(entries, page_url):
    tp_domains = set()
    for entry in entries:
        req_url = entry['request']['url']
        if is_third_party(page_url, req_url):
            tp_domains.add(get_fld(req_url))
    return tp_domains

def number_of_distinct_third_party_domains(domains):
    return len(domains)

Number of distinct third-party entities per website

In [ ]:
entities = json.loads(open("disconnect/entities.json").read())
domain_to_entity = {}
for entity_name, entity_info in entities["entities"].items():
    for domain in entity_info["properties"] + entity_info["resources"]:
        domain_to_entity[domain] = entity_name


def get_third_party_entities(domains, map):
    thirdPartyEntities = set()
    for domain in domains:
        if (domain in map):
            thirdPartyEntities.add(map[domain])
    return thirdPartyEntities

def get_number_of_distict_third_party_entities_per_website(third_party_entities):
    return len(third_party_entities)

Prepare data for plotting 

In [ ]:
# Create dummy data for trying out the plots

modes = ["desktop", "mobile", "block", "ignore"]
sites = pd.read_csv("../sites.csv")["domain"].tolist()

rows = []
for mode in modes:
    for site in sites:
        num_requests = random.randint(30, 80)
        num_third_party_requests = random.randint(30,80)
        num_dist_third_party_domains = random.randint(10,60)
        num_dist_third_party_entites_per_website = random.randint(10,60)
        rows.append({"site": site, "mode": mode, "num_requests": num_requests, "num_third_party_requests": num_third_party_requests, "num_dist_third_party_domains": num_dist_third_party_domains, 
                    "num_dist_third_party_entites_per_website": num_dist_third_party_entites_per_website})

df = pd.DataFrame(rows)



Plots for 1a - 1d

In [ ]:
sns.boxplot(df, x="mode", y="num_requests")

In [ ]:
sns.boxplot(df, x="mode", y="num_third_party_requests")

In [ ]:
sns.boxplot(df, x="mode", y="num_dist_third_party_domains")

In [ ]:
sns.boxplot(df, x="mode", y="num_dist_third_party_entites_per_website")

Create Table to compare data from 1a - 1d

In [22]:
metrics = ["num_requests", "num_third_party_requests", "num_dist_third_party_domains", "num_dist_third_party_entites_per_website"]
table = df.groupby("mode")[metrics].agg(["min", "median", "max"]).T.unstack()
table

mode                                     block              desktop         \
                                           min median   max     min median   
num_requests                              31.0   56.5  80.0    30.0   52.5   
num_third_party_requests                  32.0   56.5  80.0    30.0   51.0   
num_dist_third_party_domains              10.0   32.0  56.0    12.0   37.0   
num_dist_third_party_entites_per_website  10.0   41.5  60.0    10.0   30.0   

mode                                           ignore              mobile  \
                                           max    min median   max    min   
num_requests                              80.0   30.0   54.5  78.0   30.0   
num_third_party_requests                  80.0   30.0   57.0  80.0   30.0   
num_dist_third_party_domains              60.0   10.0   41.5  58.0   10.0   
num_dist_third_party_entites_per_website  60.0   10.0   30.5  60.0   14.0   

mode                                                   
                                         median   max  
num_requests                               53.0  80.0  
num_third_party_requests                   63.0  80.0  
num_dist_third_party_domains               35.0  60.0  
num_dist_third_party_entites_per_website   40.0  58.0